In [1]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

Mounted at /content/drive


In [ ]:
import os

ROOT = "/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research"

DIRS = {
  "shared": f"{ROOT}/shared",
  "shared_models": f"{ROOT}/shared/models",
  "shared_models_base": f"{ROOT}/shared/models/base_models",
  "shared_models_embed": f"{ROOT}/shared/models/embeddings",
  "shared_datasets": f"{ROOT}/shared/datasets",
  "shared_datasets_raw": f"{ROOT}/shared/datasets/raw",
  "shared_datasets_processed": f"{ROOT}/shared/datasets/processed",
  "shared_indexes": f"{ROOT}/shared/indexes",
  "shared_indexes_faiss": f"{ROOT}/shared/indexes/faiss",
  "shared_configs": f"{ROOT}/shared/configs",
  "experiments": f"{ROOT}/experiments",
  "pipeline": f"{ROOT}/pipeline",
  "global_results": f"{ROOT}/global_results",
  "manifests": f"{ROOT}/manifests",

  # backward-compatible aliases
  "models_base": f"{ROOT}/shared/models/base_models",
  "models_embed": f"{ROOT}/shared/models/embeddings",
  "datasets_raw": f"{ROOT}/shared/datasets/raw",
  "datasets_processed": f"{ROOT}/shared/datasets/processed",
  "indexes": f"{ROOT}/shared/indexes/faiss",
  "runs": f"{ROOT}/experiments",
}

for k, p in DIRS.items():
    os.makedirs(p, exist_ok=True)

DIRS

{'models_base': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/models/base_models',
 'models_embed': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/models/embeddings',
 'datasets_raw': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/datasets/raw',
 'datasets_processed': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/datasets/processed',
 'indexes': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss',
 'runs': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/runs',
 'manifests': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/manifests'}

In [ ]:
!pip -q install \
  "transformers>=4.43.0" \
  "datasets>=2.19.0" \
  "accelerate>=0.33.0" \
  "peft>=0.12.0" \
  "sentence-transformers>=3.0.0" \
  "faiss-cpu>=1.8.0" \
  "torch" \
  "protobuf<5" \
  "tqdm"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
BASE_MODELS = [
  {"id": "google/flan-t5-small", "save_as": "flan-t5-small", "task": "seq2seq", "enabled": True},
  {"id": "google/flan-t5-base",  "save_as": "flan-t5-base",  "task": "seq2seq", "enabled": False},
]

EMBED_MODELS = [
  {"id": "sentence-transformers/all-MiniLM-L6-v2", "save_as": "all-MiniLM-L6-v2", "enabled": True},
]

## Download Base Models (idempotent)

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

def exists_ok(path: str) -> bool:
    return os.path.isdir(path) and len(os.listdir(path)) > 0

def download_base_model(spec: dict, out_dir: str):
    model_id = spec["id"]
    save_as = spec["save_as"]
    task = spec["task"]
    enabled = spec.get("enabled", True)
    if not enabled:
        return {"model_id": model_id, "saved": False, "reason": "disabled"}

    dst = os.path.join(out_dir, save_as)
    if exists_ok(dst):
        return {"model_id": model_id, "saved": True, "path": dst, "cached": True}

    tok = AutoTokenizer.from_pretrained(model_id)
    if task == "seq2seq":
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    elif task == "causal":
        model = AutoModelForCausalLM.from_pretrained(model_id)
    else:
        raise ValueError(f"Unknown task: {task}")

    tok.save_pretrained(dst)
    model.save_pretrained(dst)
    return {"model_id": model_id, "saved": True, "path": dst, "cached": False}

base_results = [download_base_model(m, DIRS["models_base"]) for m in BASE_MODELS]
base_results

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[{'model_id': 'google/flan-t5-small',
  'saved': True,
  'path': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/models/base_models/flan-t5-small',
  'cached': False},
 {'model_id': 'google/flan-t5-base', 'saved': False, 'reason': 'disabled'}]

## Download Embedding Models (idempotent)

In [ ]:
from sentence_transformers import SentenceTransformer

def download_embed_model(spec: dict, out_dir: str):
    model_id = spec["id"]
    save_as = spec["save_as"]
    enabled = spec.get("enabled", True)
    if not enabled:
        return {"model_id": model_id, "saved": False, "reason": "disabled"}

    dst = os.path.join(out_dir, save_as)
    if exists_ok(dst):
        return {"model_id": model_id, "saved": True, "path": dst, "cached": True}

    m = SentenceTransformer(model_id)
    m.save(dst)
    return {"model_id": model_id, "saved": True, "path": dst, "cached": False}

embed_results = [download_embed_model(m, DIRS["models_embed"]) for m in EMBED_MODELS]
embed_results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[{'model_id': 'sentence-transformers/all-MiniLM-L6-v2',
  'saved': True,
  'path': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/models/embeddings/all-MiniLM-L6-v2',
  'cached': False}]

##  Dataset Downloads

### Dolly 15k + Dolly small subset

In [ ]:
from datasets import load_dataset, DatasetDict
import os

def save_dataset(name: str, ds: DatasetDict, out_dir: str):
    dst = os.path.join(out_dir, name)
    if os.path.exists(dst) and len(os.listdir(dst)) > 0:
        return {"name": name, "saved": True, "path": dst, "cached": True}
    ds.save_to_disk(dst)
    return {"name": name, "saved": True, "path": dst, "cached": False}

dolly = load_dataset("databricks/databricks-dolly-15k")
res_dolly = save_dataset("dolly_15k", dolly, DIRS["shared_datasets_raw"])

small = DatasetDict({"train": dolly["train"].shuffle(seed=42).select(range(1000))})
res_small = save_dataset("dolly_small_1k", small, DIRS["shared_datasets_raw"])

res_dolly, res_small

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15011 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

({'name': 'dolly_15k',
  'saved': True,
  'path': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/datasets/raw/dolly_15k',
  'cached': False},
 {'name': 'dolly_small_1k',
  'saved': True,
  'path': '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/datasets/raw/dolly_small_1k',
  'cached': False})

## Write Bootstrap Manifest

In [ ]:
import json, time

manifest = {
  "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
  "root": ROOT,
  "dirs": DIRS,
  "base_models": base_results,
  "embed_models": embed_results,
  "datasets": [res_dolly, res_small],
}

manifest_path = f"{DIRS['manifests']}/bootstrap_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Wrote:", manifest_path)

Wrote: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/manifests/bootstrap_manifest.json
